In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import bbknn
import scrublet as scr
import matplotlib.pyplot as plt
import scanpy.external as sce
import hotspot

sc.settings.set_figure_params(dpi=1000,figsize=(5, 5))
sc.logging.print_header()

In [ ]:
## data dir
origin_dir = " "
all_meta_dir = " "
output_dir = " "
all_cell_type = ["T&NK_cell","B_cell","Endothelia","Fibroblast","Myeloid_cell","Neutrophils","Plasma_cell"]
# all_cell_type = ["Neutrophils"]

## pre work
regress = True
cluster_method = "leiden"
bbknn_ridge = True
resolution = 2
random_state = 123
n_iterations = -1

In [ ]:
# all meta read
all_meta = pd.read_csv(f"{all_meta_dir}/all_cell_combine_meta.csv")

In [ ]:
all_results = []
for celltype in all_cell_type:

    print(f"################################## {celltype} process ##################################")

    ## data read
    if celltype == "T&NK_cell":
        scRNA_current = []
        scRNA_current1 = sc.read_h5ad(f"{origin_dir}/T_cell/scRNA_gene_filter.h5ad")
        scRNA_current2 = sc.read_h5ad(f"{origin_dir}/NK_cell/scRNA_gene_filter.h5ad")
        scRNA_current = [scRNA_current1,scRNA_current2]
        scRNA_current = sc.concat(scRNA_current)
        scRNA_current.obs_names_make_unique()

        del scRNA_current1
        del scRNA_current2
        gc.collect()

    else:
        scRNA_current = sc.read_h5ad(f"{origin_dir}/{celltype}/scRNA_gene_filter.h5ad")

    ## join final cell type meta
    scRNA_current.obs = pd.merge(scRNA_current.obs, all_meta[['cell_id', 'final_cell_type']], on='cell_id', how='left')

    ## data nor
    scRNA_current.layers["counts"] = scRNA_current.X.copy()
    sc.pp.normalize_total(scRNA_current, target_sum=1e4)
    sc.pp.log1p(scRNA_current)
    scRNA_current.layers["log1p"] = scRNA_current.X.copy()
    scRNA_current.raw = scRNA_current

    ## DEG find
    sc.tl.rank_genes_groups(scRNA_current, "final_cell_type", method="wilcoxon", pts=True)

    ## DEG filter
    groups = scRNA_current.uns['rank_genes_groups']['names'].dtype.names
    all_dfs = []
    for group in groups:
        df = sc.get.rank_genes_groups_df(scRNA_current, group=group)
        df['group'] = group
        all_dfs.append(df)

    ## current DEG combine
    current_results = pd.concat(all_dfs,axis=0)

    ## current save
    current_results.to_csv(f'{output_dir}/{celltype}_DEG.csv', index=False)

    ## all DEG combine
    all_results.append(current_results)

    ## space free
    del scRNA_current
    gc.collect()

## data save
all_results = pd.concat(all_results,axis=0)
all_results.to_csv(f'{output_dir}/all_DEG.csv', index=False)

In [ ]:
all_results